# Step 1: Load Dataset

In [ ]:
import kagglehub
import os

path = kagglehub.dataset_download(
    "uciml/sms-spam-collection-dataset"
)

print("Dataset path:", path)

print(os.listdir(path))

Using Colab cache for faster access to the 'sms-spam-collection-dataset' dataset.
Dataset path: /kaggle/input/sms-spam-collection-dataset
['spam.csv']


In [ ]:
from kagglehub import KaggleDatasetAdapter
import kagglehub

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "uciml/sms-spam-collection-dataset",
    'spam.csv',
    pandas_kwargs={'encoding': 'latin-1'}
)

print(df.head())

/tmp/ipykernel_1882/1813433042.py:4: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'sms-spam-collection-dataset' dataset.
     v1                                                 v2 Unnamed: 2  \
0   ham  Go until jurong point, crazy.. Available only ...        NaN   
1   ham                      Ok lar... Joking wif u oni...        NaN   
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...        NaN   
3   ham  U dun say so early hor... U c already then say...        NaN   
4   ham  Nah I don't think he goes to usf, he lives aro...        NaN   

  Unnamed: 3 Unnamed: 4  
0        NaN        NaN  
1        NaN        NaN  
2        NaN        NaN  
3        NaN        NaN  
4        NaN        NaN  


# Step 2: Explore Dataset

In [ ]:
print(df.shape)

print(df.columns)

print(df.info())

print(df.isnull().sum())

(5572, 5)
Index(['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   v1          5572 non-null   object
 1   v2          5572 non-null   object
 2   Unnamed: 2  50 non-null     object
 3   Unnamed: 3  12 non-null     object
 4   Unnamed: 4  6 non-null      object
dtypes: object(5)
memory usage: 217.8+ KB
None
v1               0
v2               0
Unnamed: 2    5522
Unnamed: 3    5560
Unnamed: 4    5566
dtype: int64


# Step 3: Rename Columns

In [ ]:
df = df.rename(columns={
    'v1':'label',
    'v2':'message'
})

print(df.head())

  label                                            message Unnamed: 2  \
0   ham  Go until jurong point, crazy.. Available only ...        NaN   
1   ham                      Ok lar... Joking wif u oni...        NaN   
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...        NaN   
3   ham  U dun say so early hor... U c already then say...        NaN   
4   ham  Nah I don't think he goes to usf, he lives aro...        NaN   

  Unnamed: 3 Unnamed: 4  
0        NaN        NaN  
1        NaN        NaN  
2        NaN        NaN  
3        NaN        NaN  
4        NaN        NaN  


# Step 4: Convert Labels to Numbers

In [ ]:
df['label'] = df['label'].map({
    'ham':0,
    'spam':1
})

print(df.head())

   label                                            message Unnamed: 2  \
0      0  Go until jurong point, crazy.. Available only ...        NaN   
1      0                      Ok lar... Joking wif u oni...        NaN   
2      1  Free entry in 2 a wkly comp to win FA Cup fina...        NaN   
3      0  U dun say so early hor... U c already then say...        NaN   
4      0  Nah I don't think he goes to usf, he lives aro...        NaN   

  Unnamed: 3 Unnamed: 4  
0        NaN        NaN  
1        NaN        NaN  
2        NaN        NaN  
3        NaN        NaN  
4        NaN        NaN  


# Step 5: Check Class Distribution

In [ ]:
print(df['label'].value_counts())

label
0    4825
1     747
Name: count, dtype: int64


# Step 6: Text Preprocessing

In [ ]:
import re

def clean_text(text):

    text = text.lower()

    text = re.sub(r'[^a-zA-Z ]','',text)

    return text

df['message'] = df['message'].apply(clean_text)

# Step 7: Convert Text into Numerical Features

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(df['message'])

y = df['label']

# Step 8: Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Step 9: Train Model
Naive Bayes

Best model for spam detection.

In [ ]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()

model.fit(X_train,y_train)

MultinomialNB()

# Step 10: Prediction

In [ ]:
y_pred = model.predict(X_test)

print(y_pred[:10])

[0 0 0 0 1 0 0 0 0 0]


# Step 11: Evaluate Model

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

accuracy = accuracy_score(y_test,y_pred)

print("Accuracy:",accuracy)

print("\nClassification Report")
print(classification_report(y_test,y_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test,y_pred))

Accuracy: 0.9506726457399103

Classification Report
              precision    recall  f1-score   support

           0       0.95      1.00      0.97       965
           1       1.00      0.63      0.78       150

    accuracy                           0.95      1115
   macro avg       0.97      0.82      0.87      1115
weighted avg       0.95      0.95      0.95      1115


Confusion Matrix
[[965   0]
 [ 55  95]]


# Step 12: Test Custom Message

In [ ]:
sample = ["Congratulations! You won a free iPhone. Click here now"]

sample_vector = vectorizer.transform(sample)

prediction = model.predict(sample_vector)

if prediction[0] == 1:
    print("Spam")
else:
    print("Ham")

Spam
